## Installing dependencies 

In [ ]:
# pip install pandas numpy openpyxl scikit-learn xgboost shap gensim matplotlib seaborn

In [ ]:
# importing the necessary libraries 
import pandas as pd
import numpy as np
import re
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

#### NLP & Topic Modeling 

In [ ]:
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

#### Machine Learning 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             precision_score, recall_score, confusion_matrix,
                             roc_curve, precision_recall_curve,
                             average_precision_score, brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler

#### Explainability 

In [ ]:
import shap

In [ ]:
import matplotlib
#matplotlib.use('Agg')
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11, 'figure.dpi': 180,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#888', 'axes.linewidth': 0.5, 'figure.facecolor': 'white'
})

#### Configuration

In [ ]:
INPUT_FILE = 'thesis_BDU_cleaned.xlsx'  # Update path as needed
OUTPUT_DIR = 'model_figures'
RANDOM_STATE = 42
LDA_BEST_K = 9           # Selected by coherence testing (c_v = 0.454)
CV_FOLDS = 5
CV_REPEATS = 3

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Stage 1: Loading the data

In [ ]:
print("=" * 60)
print("STAGE 1: Loading data")
print("=" * 60)

bd_data = pd.read_excel(INPUT_FILE)
print(f"Loaded: {bd_data.shape[0]} records, {bd_data.shape[1]} columns")

In [ ]:
bd_data.head()

## Stage 2: NLP Preprocessing

In [ ]:
print("\n" + "=" * 60)
print("STAGE 2: NLP preprocessing")
print("=" * 60)

In [ ]:
# Domain-specific stopwords (procurement/consulting boilerplate that adds
# no discriminative value for topic separation in this corpus)
domain_stops = {
    'shall', 'must', 'may', 'would', 'could', 'should', 'will',
    'consultant', 'consultancy', 'consulting', 'contractor', 'service', 'services',
    'proposal', 'request', 'rfp', 'rfi', 'rfq', 'tender', 'bid',
    'provide', 'provided', 'providing', 'include', 'including', 'includes',
    'required', 'requirement', 'requirements', 'expected', 'ensure',
    'work', 'working', 'project', 'assignment', 'activity', 'activities',
    'organization', 'organisation', 'company', 'firm', 'client',
    'submission', 'submit', 'submitted', 'deadline', 'date',
    'also', 'well', 'based', 'related', 'relevant', 'key',
    'support', 'supporting', 'assist', 'assistance',
    'development', 'develop', 'developing',
    'management', 'manage', 'managing',
    'implementation', 'implement', 'implementing',
    'assessment', 'assess', 'assessing',
    'analysis', 'analyze', 'analyse', 'analyzing',
    'report', 'reporting', 'reports',
    'capacity', 'building', 'strengthening',
    'national', 'international', 'global', 'local', 'regional',
    'specific', 'overall', 'following', 'within', 'across',
    'term', 'terms', 'reference', 'scope',
    'day', 'days', 'month', 'months', 'year', 'years', 'period',
    'please', 'note', 'annex', 'attachment', 'page', 'section',
    'eg', 'ie', 'etc', 'thereof', 'herein',
}
all_stops = STOPWORDS.union(domain_stops)

In [ ]:
def preprocess_text(text):
    """Tokenise, lowercase, remove non-alpha, filter stopwords and short tokens."""
    if pd.isna(text):
        return []
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = simple_preprocess(text, deacc=True, min_len=3, max_len=30)
    tokens = [t for t in tokens if t not in all_stops and len(t) > 2]
    return tokens

bd_data['tokens'] = bd_data['text_for_nlp'].apply(preprocess_text)
print(f"Tokenised: mean={bd_data['tokens'].str.len().mean():.0f} tokens/doc")

# Build dictionary from documents with sufficient content
MIN_TOKENS = 5
good_mask = bd_data['tokens'].str.len() >= MIN_TOKENS
dictionary = corpora.Dictionary(bd_data.loc[good_mask, 'tokens'].tolist())
dictionary.filter_extremes(no_below=5, no_above=0.6)
print(f"Dictionary: {len(dictionary)} tokens after filtering")

## Stage 3: LDA Topic Modelling

In [ ]:
print("\n" + "=" * 60)
print("STAGE 3: LDA topic modelling")
print("=" * 60)

In [ ]:
# Coherence testing across topic counts
corpus_train = [dictionary.doc2bow(doc) for doc in bd_data.loc[good_mask, 'tokens'].tolist()]

print("Coherence testing (k=4..11):")
coherence_results = []
for k in range(4, 12):
    lda = LdaModel(corpus=corpus_train, id2word=dictionary, num_topics=k,
                    random_state=RANDOM_STATE, passes=15, alpha='auto', eta='auto',
                    chunksize=100, per_word_topics=True)
    coh = CoherenceModel(model=lda, texts=bd_data.loc[good_mask, 'tokens'].tolist(),
                         dictionary=dictionary, coherence='c_v')
    score = coh.get_coherence()
    coherence_results.append({'k': k, 'coherence': score})
    print(f"  k={k:2d} | c_v={score:.4f}")

best_k = max(coherence_results, key=lambda x: x['coherence'])['k']
print(f"Best k: {best_k} (c_v={max(coherence_results, key=lambda x: x['coherence'])['coherence']:.4f})")

# Train final model with best k
lda_model = LdaModel(corpus=corpus_train, id2word=dictionary, num_topics=best_k,
                      random_state=RANDOM_STATE, passes=20, alpha='auto', eta='auto',
                      chunksize=100, per_word_topics=True)

In [ ]:
# Extract topic distributions for ALL records
corpus_all = [dictionary.doc2bow(doc) for doc in bd_data['tokens'].tolist()]
topic_dists = []
for bow in corpus_all:
    if len(bow) == 0:
        topic_dists.append([0.0] * best_k)
    else:
        doc_topics = lda_model.get_document_topics(bow, minimum_probability=0.0)
        dist = [0.0] * best_k
        for topic_id, prob in doc_topics:
            dist[topic_id] = prob
        topic_dists.append(dist)

topic_cols = [f'topic_{i}' for i in range(best_k)]
for i, col in enumerate(topic_cols):
    bd_data[col] = [d[i] for d in topic_dists]
bd_data['dominant_topic'] = pd.DataFrame(topic_dists).idxmax(axis=1).values

# Save LDA artifacts
lda_model.save('lda_model')
dictionary.save('lda_dictionary')

In [ ]:
print("\nTopics:")
topic_labels = [
    'Digital payments & banking', 'Risk & compliance', 'Strategy & planning',
    'MSME & agricultural finance', 'Value chain & trade', 'Inclusion, gender & energy',
    'Digital literacy & cash', 'Fund & investment mgmt', 'Capital markets & green'
]
for idx in range(best_k):
    words = ', '.join([w[0] for w in lda_model.show_topic(idx, topn=8)])
    print(f"  T{idx} ({topic_labels[idx]}): {words}")

## Stage 4: Feature Matrix Construction

In [ ]:
print("\n" + "=" * 60)
print("STAGE 4: Feature matrix construction")
print("=" * 60)

In [ ]:
model_data = bd_data[bd_data['target_win'].notna()].copy()
print(f"Modelling subset: {len(model_data)} records ({model_data['target_win'].sum():.0f} wins)")

In [ ]:
# Region grouping
def group_region(r):
    if pd.isna(r): return 'Unknown'
    if r in ['East Africa', 'Southern Africa', 'West Africa']: return r
    if r in ['Africa']: return 'Multi-country/Global'
    return 'Other'

model_data['region_group'] = model_data['primary_region'].apply(group_region)
model_data['modality_clean'] = model_data['Modality of Submission'].fillna('Unknown')
model_data['modality_clean'] = model_data['modality_clean'].replace(
    {'Physical': 'Other', 'Postal mail (Envelop)': 'Other'})

In [ ]:
# Handle missing values
model_data['pub_to_deadline_missing'] = model_data['pub_to_deadline_days'].isna().astype(int)
model_data['pub_to_deadline_days'] = model_data['pub_to_deadline_days'].fillna(
    model_data['pub_to_deadline_days'].median())
model_data['lead_time_days'] = model_data['lead_time_days'].fillna(
    model_data['lead_time_days'].median())

In [ ]:
model_data.tail(10)

In [ ]:
# Build feature matrix
numeric_feats = ['value_range_ordinal', 'lead_time_days', 'pub_to_deadline_days',
                 'client_opportunity_count', 'is_repeat_client', 'is_multi_region',
                 'created_quarter', 'has_usable_summary', 'pub_to_deadline_missing']

X_numeric = model_data[numeric_feats + topic_cols].copy()
X_dummies = pd.get_dummies(pd.DataFrame({
    'region_group': model_data['region_group'].values,
    'currency_group': model_data['currency_group'].fillna('Unknown').values,
    'modality_clean': model_data['modality_clean'].values
}), drop_first=True, dtype=int)

X = pd.concat([X_numeric.reset_index(drop=True), X_dummies.reset_index(drop=True)], axis=1)
X.columns = [c.replace(' ', '_').replace('[', '').replace(']', '').replace('<', '') for c in X.columns]
y = model_data['target_win'].values.astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix: {X.shape[0]} samples x {X.shape[1]} features")

## Stage 5: Model Training and Evaluation

In [ ]:
print("\n" + "=" * 60)
print("STAGE 5: Model training and evaluation")
print("=" * 60)

In [ ]:
scale_pos = (1 - y).sum() / y.sum()

models = {
    'Logistic Regression': LogisticRegression(
        C=1.0, penalty='l2', class_weight='balanced',
        max_iter=1000, random_state=RANDOM_STATE, solver='lbfgs'),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1,
        scale_pos_weight=scale_pos, min_child_weight=3,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0),
}

cv = RepeatedStratifiedKFold(n_splits=CV_FOLDS, n_repeats=CV_REPEATS, random_state=RANDOM_STATE)
results = {}
cv_probas = {}

for name, model in models.items():
    X_use = X_scaled if name == 'Logistic Regression' else X.values
    aucs, f1s, precs, recs, accs, briers = [], [], [], [], [], []

    for train_idx, test_idx in cv.split(X_use, y):
        m = pickle.loads(pickle.dumps(model))
        m.fit(X_use[train_idx], y[train_idx])
        y_pred = m.predict(X_use[test_idx])
        y_proba = m.predict_proba(X_use[test_idx])[:, 1]
        aucs.append(roc_auc_score(y[test_idx], y_proba))
        f1s.append(f1_score(y[test_idx], y_pred))
        precs.append(precision_score(y[test_idx], y_pred, zero_division=0))
        recs.append(recall_score(y[test_idx], y_pred))
        accs.append(accuracy_score(y[test_idx], y_pred))
        briers.append(brier_score_loss(y[test_idx], y_proba))

    results[name] = {
        'AUC-ROC': (np.mean(aucs), np.std(aucs)),
        'F1': (np.mean(f1s), np.std(f1s)),
        'Precision': (np.mean(precs), np.std(precs)),
        'Recall': (np.mean(recs), np.std(recs)),
        'Accuracy': (np.mean(accs), np.std(accs)),
        'Brier Score': (np.mean(briers), np.std(briers)),
    }

    # Single-pass CV predictions for plotting
    cv_single = RepeatedStratifiedKFold(n_splits=CV_FOLDS, n_repeats=1, random_state=RANDOM_STATE)
    cv_probas[name] = cross_val_predict(model, X_use, y, cv=cv_single, method='predict_proba')[:, 1]

    print(f"\n{name}:")
    for metric, (mean, std) in results[name].items():
        print(f"  {metric:15s}: {mean:.3f} +/- {std:.3f}")

In [ ]:
# Retrain on full data
final_models = {}
for name, model in models.items():
    X_use = X_scaled if name == 'Logistic Regression' else X.values
    model.fit(X_use, y)
    final_models[name] = model

## Stage 6: Evaluation Plots

In [ ]:
print("\n" + "=" * 60)
print("STAGE 6: Generating evaluation plots")
print("=" * 60)

In [ ]:
model_colors = {'Logistic Regression': '#2E5090', 'Random Forest': '#5DCAA5', 'XGBoost': '#D85A30'}

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(6, 5))
for name, probas in cv_probas.items():
    fpr, tpr, _ = roc_curve(y, probas)
    auc = roc_auc_score(y, probas)
    ax.plot(fpr, tpr, color=model_colors[name], linewidth=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1], [0,1], 'k--', linewidth=0.8, alpha=0.4)
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('ROC curves (cross-validated)', fontweight='bold', fontsize=13)
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout(); fig.savefig(f'{OUTPUT_DIR}/roc_curves.png', bbox_inches='tight'); plt.show(); plt.close()

In [ ]:
# Precision-Recall curves
fig, ax = plt.subplots(figsize=(6, 5))
for name, probas in cv_probas.items():
    prec, rec, _ = precision_recall_curve(y, probas)
    ap = average_precision_score(y, probas)
    ax.plot(rec, prec, color=model_colors[name], linewidth=2, label=f'{name} (AP={ap:.3f})')
ax.axhline(y=y.mean(), color='gray', linestyle='--', linewidth=0.8, alpha=0.5, label=f'Baseline ({y.mean():.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-recall curves (cross-validated)', fontweight='bold', fontsize=13)
ax.legend(fontsize=9); plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/pr_curves.png', bbox_inches='tight'); plt.show(); plt.close()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for idx, (name, probas) in enumerate(cv_probas.items()):
    y_pred = (probas >= 0.5).astype(int)
    cm = confusion_matrix(y, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Lost', 'Won'], yticklabels=['Lost', 'Won'],
                cbar=False, annot_kws={'size': 14, 'fontweight': 'bold'})
    axes[idx].set_xlabel('Predicted'); axes[idx].set_ylabel('Actual')
    axes[idx].set_title(name, fontweight='bold', fontsize=11)
plt.suptitle('Confusion matrices (threshold=0.5)', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout(); fig.savefig(f'{OUTPUT_DIR}/confusion_matrices.png', bbox_inches='tight'); plt.show(); plt.close()

In [ ]:
# Calibration plot
fig, ax = plt.subplots(figsize=(6, 5))
for name, probas in cv_probas.items():
    fraction_pos, mean_predicted = calibration_curve(y, probas, n_bins=8, strategy='quantile')
    ax.plot(mean_predicted, fraction_pos, 's-', color=model_colors[name], linewidth=1.5, markersize=5, label=name)
ax.plot([0,1], [0,1], 'k--', linewidth=0.8, alpha=0.4, label='Perfect')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration plot', fontweight='bold', fontsize=13)
ax.legend(fontsize=9); plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/calibration_plot.png', bbox_inches='tight'); plt.show(); plt.close()

print("Evaluation plots saved.")

## Stage 7: SHAP Analysis

On best model: Logistic Regression

In [ ]:
print("\n" + "=" * 60)
print("STAGE 7: SHAP analysis")
print("=" * 60)

In [ ]:
# Renaming derived features
feature_names_pretty = {
    'value_range_ordinal': 'Contract value band', 'lead_time_days': 'Lead time (days)',
    'pub_to_deadline_days': 'Publication window (days)', 'client_opportunity_count': 'Client opp. count',
    'is_repeat_client': 'Repeat client', 'is_multi_region': 'Multi-region',
    'created_quarter': 'Quarter created', 'has_usable_summary': 'Has TOR/RFP summary',
    'pub_to_deadline_missing': 'Pub. date missing',
    'topic_0': 'T0: Digital payments', 'topic_1': 'T1: Risk & compliance',
    'topic_2': 'T2: Strategy & planning', 'topic_3': 'T3: MSME & agri finance',
    'topic_4': 'T4: Value chain & trade', 'topic_5': 'T5: Inclusion & gender',
    'topic_6': 'T6: Digital literacy', 'topic_7': 'T7: Fund & investment',
    'topic_8': 'T8: Capital markets',
    'region_group_Multi-country/Global': 'Region: Multi-country',
    'region_group_Other': 'Region: Other', 'region_group_Southern_Africa': 'Region: Southern Africa',
    'region_group_Unknown': 'Region: Unknown', 'region_group_West_Africa': 'Region: West Africa',
    'currency_group_GBP': 'Currency: GBP', 'currency_group_KSh': 'Currency: KSh',
    'currency_group_Other': 'Currency: Other', 'currency_group_USD': 'Currency: USD',
    'currency_group_Unknown': 'Currency: Unknown', 'currency_group_ZAR': 'Currency: ZAR',
    'modality_clean_Other': 'Modality: Other', 'modality_clean_Portal': 'Modality: Portal',
    'modality_clean_Unknown': 'Modality: Unknown',
}

In [ ]:
X_display = X.copy()
X_display.columns = [feature_names_pretty.get(c, c) for c in X.columns]
X_scaled_bd_data = pd.DataFrame(X_scaled, columns=X_display.columns)

lr_model = final_models['Logistic Regression']
explainer = shap.LinearExplainer(lr_model, X_scaled_bd_data)
shap_values = explainer.shap_values(X_scaled_bd_data)

In [ ]:
# Beeswarm
fig, ax = plt.subplots(figsize=(9, 8))
shap.summary_plot(shap_values, X_display, show=False, max_display=20, plot_size=None)
plt.title('SHAP feature importance (Logistic Regression)', fontweight='bold', fontsize=13, pad=15)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/shap_beeswarm.png', bbox_inches='tight'); plt.show(); plt.close()

In [ ]:
# Top-10 bar
fig, ax = plt.subplots(figsize=(7, 5))
shap.summary_plot(shap_values, X_display, plot_type='bar', show=False, max_display=10, plot_size=None)
plt.title('Top-10 features by mean |SHAP value|', fontweight='bold', fontsize=13, pad=15)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/shap_top10_bar.png', bbox_inches='tight'); plt.show(); plt.close()

In [ ]:
# Waterfall plots for 3 cases
lr_probas = lr_model.predict_proba(X_scaled)[:, 1]
predictions = (lr_probas >= 0.5).astype(int)

In [ ]:
cases = []
# Case 1: Best correctly predicted win
win_idx = np.where(y == 1)[0][np.argmax(lr_probas[y == 1])]
cases.append((win_idx, "Correctly predicted win"))
# Case 2: Best correctly predicted loss
loss_idx = np.where(y == 0)[0][np.argmin(lr_probas[y == 0])]
cases.append((loss_idx, "Correctly predicted loss"))
# Case 3: Most interesting false negative
fn_mask = (y == 1) & (predictions == 0)
if fn_mask.sum() > 0:
    fn_idx = np.where(fn_mask)[0][np.argmax(lr_probas[fn_mask])]
    cases.append((fn_idx, "False negative (won, predicted loss)"))

shap_explanation = shap.Explanation(values=shap_values, base_values=explainer.expected_value,
                                    data=X_display.values, feature_names=X_display.columns.tolist())

for case_num, (idx, label) in enumerate(cases, 1):
    fig, ax = plt.subplots(figsize=(8, 6))
    shap.waterfall_plot(shap_explanation[idx], max_display=12, show=False)
    actual = "Won" if y[idx] == 1 else "Lost"
    plt.title(f'Case {case_num}: {label}\nActual={actual}, P(win)={lr_probas[idx]:.3f}',
              fontweight='bold', fontsize=11, pad=15)
    plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/shap_waterfall_{case_num}.png', bbox_inches='tight'); plt.close()

print("SHAP plots saved.")

## Stage 8: Score all Records

In [ ]:
print("\n" + "=" * 60)
print("STAGE 8: Scoring all records")
print("=" * 60)

In [ ]:
# Rebuild feature matrix for all 439 records (same transformations)
all_bd_data = bd_data.copy()
all_bd_data['region_group'] = all_bd_data['primary_region'].apply(group_region)
all_bd_data['modality_clean'] = all_bd_data['Modality of Submission'].fillna('Unknown')
all_bd_data['modality_clean'] = all_bd_data['modality_clean'].replace(
    {'Physical': 'Other', 'Postal mail (Envelop)': 'Other'})
all_bd_data['pub_to_deadline_missing'] = all_bd_data['pub_to_deadline_days'].isna().astype(int)
all_bd_data['pub_to_deadline_days'] = all_bd_data['pub_to_deadline_days'].fillna(
    model_data['pub_to_deadline_days'].median())
all_bd_data['lead_time_days'] = all_bd_data['lead_time_days'].fillna(all_bd_data['lead_time_days'].median())

X_all_num = all_bd_data[numeric_feats + topic_cols].copy()
X_all_dum = pd.get_dummies(pd.DataFrame({
    'region_group': all_bd_data['region_group'].values,
    'currency_group': all_bd_data['currency_group'].fillna('Unknown').values,
    'modality_clean': all_bd_data['modality_clean'].values
}), drop_first=True, dtype=int)
X_all = pd.concat([X_all_num.reset_index(drop=True), X_all_dum.reset_index(drop=True)], axis=1)
X_all.columns = [c.replace(' ', '_').replace('[', '').replace(']', '').replace('<', '') for c in X_all.columns]
for col in X.columns:
    if col not in X_all.columns:
        X_all[col] = 0
X_all = X_all[X.columns]

all_bd_data['win_probability'] = lr_model.predict_proba(scaler.transform(X_all))[:, 1]


In [ ]:
# Export
export_cols = ['Id', 'Project_title', 'Status', 'Practice Decision', 'Stage',
               'RF Due Date', 'RF Outcome', 'Country', 'Region', 'primary_region',
               'Value range', 'Currency', 'h_ClientName', 'Client',
               'Modality of Submission', 'Created', 'Summary',
               'target_win', 'target_go', 'win_probability',
               'dominant_topic', 'lead_time_days', 'client_opportunity_count',
               'is_repeat_client', 'is_multi_region', 'value_range_ordinal',
               'currency_group', 'has_usable_summary'] + topic_cols

all_bd_data[export_cols].to_excel('thesis_scored_opportunities.xlsx', index=False)

print(f"Scored {len(all_bd_data)} records. Output: thesis_scored_opportunities.xlsx")
print(f"\nPIPELINE COMPLETE.")

# =============================================================================
# SESSION 2 EXTENSION: STRATEGIC VALUE FRAMEWORK AND SCENARIO SCORING
# =============================================================================

# This script continues the bdu_data_modelling_win_probabilities pipeline to compute strategic value scores across four dimensions, combine them with win probability under three scenarios, and produce Power BI-ready outputs.
#
# Maintaining the variables defined previously: bd_data, model_data, all_bd_data, lr_model, scaler, X.columns, topic_cols.

In [ ]:
print("\n" + "=" * 60)
print("STAGE 9: Strategic value scoring (4 dimensions)")
print("=" * 60)

In [ ]:
# Use all_bd_data which has win_probability already scored
all_bd_data = pd.read_excel('thesis_scored_opportunities.xlsx') if 'win_probability' not in all_bd_data.columns else all_bd_data

In [ ]:
all_bd_data.head()

## Dimension 1: Contract Scale & Value

In [ ]:
# Ordinal mapping from value band to [0, 1]
value_map = {0: 0.0, 1: 0.20, 2: 0.40, 3: 0.60, 4: 0.80, 5: 1.0}
all_bd_data['score_value'] = all_bd_data['value_range_ordinal'].map(value_map).fillna(0.0)

## Dimension 2: Strategic Client/Funder Importance

In [ ]:
# Composite of client volume and (shrunken) client win rate
resolved = all_bd_data[all_bd_data['target_win'].notna()]
overall_win_rate = resolved['target_win'].mean()

client_volume = all_bd_data.groupby('Client').size().to_dict()
client_stats = resolved.groupby('Client').agg(wins=('target_win','sum'), total=('target_win','count')).reset_index()
# Shrinkage with prior n=3 at overall mean stabilises small-sample clients
client_stats['shrunken_win_rate'] = (client_stats['wins'] + overall_win_rate * 3) / (client_stats['total'] + 3)
client_win_rate = dict(zip(client_stats['Client'], client_stats['shrunken_win_rate']))

max_log_vol = np.log1p(max(client_volume.values()))
all_bd_data['client_volume_norm'] = all_bd_data['Client'].map(client_volume).apply(lambda v: np.log1p(v) / max_log_vol)
all_bd_data['client_winrate_shrunk'] = all_bd_data['Client'].map(client_win_rate).fillna(overall_win_rate)
# 60% volume + 40% shrunken win rate
all_bd_data['score_client'] = 0.6 * all_bd_data['client_volume_norm'] + 0.4 * all_bd_data['client_winrate_shrunk']

## Dimension 3: Practice Growth Alignment (data-derived)

In [ ]:
# Compute topic-level win rate signals weighted by topic probability
topic_win_signal = {}
for t in topic_cols:
    weighted_wins = (resolved[t] * resolved['target_win']).sum()
    weighted_total = resolved[t].sum()
    topic_wr = weighted_wins / weighted_total if weighted_total > 0 else overall_win_rate
    # Floor at 70% of baseline so underperforming topics score 0
    topic_win_signal[t] = max(0, topic_wr - overall_win_rate * 0.7)

max_topic_signal = max(topic_win_signal.values())
if max_topic_signal > 0:
    topic_win_signal = {k: v/max_topic_signal for k, v in topic_win_signal.items()}

# Opportunity's topic-based growth fit (weighted sum across topic probabilities)
all_bd_data['growth_topic_fit'] = 0.0
for t in topic_cols:
    all_bd_data['growth_topic_fit'] += all_bd_data[t] * topic_win_signal[t]

# Region-level win rate signals (with shrinkage)
region_stats = resolved.groupby('primary_region').agg(wins=('target_win','sum'), total=('target_win','count')).reset_index()
region_stats['shrunken_wr'] = (region_stats['wins'] + overall_win_rate * 3) / (region_stats['total'] + 3)
# Only trust regions with >= 4 observations; default small-n to baseline/2
region_stats['signal'] = np.where(region_stats['total'] >= 4,
                                   np.maximum(0, region_stats['shrunken_wr'] - overall_win_rate * 0.7),
                                   overall_win_rate * 0.5)
max_region_signal = region_stats['signal'].max()
region_stats['signal_norm'] = region_stats['signal'] / max_region_signal if max_region_signal > 0 else 0
region_signal = dict(zip(region_stats['primary_region'], region_stats['signal_norm']))
all_bd_data['growth_region_fit'] = all_bd_data['primary_region'].map(region_signal).fillna(0.3)

# Final growth alignment: 70% topic fit + 30% region fit, then normalise to [0, 1]
raw_growth = 0.7 * all_bd_data['growth_topic_fit'] + 0.3 * all_bd_data['growth_region_fit']
all_bd_data['score_growth'] = (raw_growth - raw_growth.min()) / (raw_growth.max() - raw_growth.min())

## Dimension 4: Portfolio DIversification

In [ ]:
# Inverse of log-normalised client opportunity count
max_count = all_bd_data['client_opportunity_count'].max()
all_bd_data['score_diversification'] = 1 - (np.log1p(all_bd_data['client_opportunity_count']) / np.log1p(max_count))

## Scenario COnfiguration

In [ ]:
scenarios = {
    'Win-maximising': {'w_win': 0.70, 'w_strategic': 0.30,
                       'w_value': 0.25, 'w_client': 0.35, 'w_growth': 0.25, 'w_diversification': 0.15},
    'Balanced': {'w_win': 0.50, 'w_strategic': 0.50,
                 'w_value': 0.25, 'w_client': 0.25, 'w_growth': 0.25, 'w_diversification': 0.25},
    'Growth-maximising': {'w_win': 0.30, 'w_strategic': 0.70,
                          'w_value': 0.20, 'w_client': 0.10, 'w_growth': 0.40, 'w_diversification': 0.30},
}

# Compute composite score and rank under each scenario
for name, cfg in scenarios.items():
    all_bd_data[f'strategic_value_{name}'] = (
        cfg['w_value'] * all_bd_data['score_value'] +
        cfg['w_client'] * all_bd_data['score_client'] +
        cfg['w_growth'] * all_bd_data['score_growth'] +
        cfg['w_diversification'] * all_bd_data['score_diversification']
    )
    all_bd_data[f'composite_score_{name}'] = (
        cfg['w_win'] * all_bd_data['win_probability'] +
        cfg['w_strategic'] * all_bd_data[f'strategic_value_{name}']
    )
    all_bd_data[f'rank_{name}'] = all_bd_data[f'composite_score_{name}'].rank(ascending=False, method='min').astype(int)

print(f"Scored {len(all_bd_data)} records under 3 scenarios")


In [ ]:
# Save final output
all_bd_data.to_excel('thesis_scored_opportunities_final.xlsx', index=False)
print("Final output: thesis_scored_opportunities_final.xlsx")
print("\nPIPELINE COMPLETE.")